# Hybrid RAG Pipeline on the OrchestrAI Synthetic Enterprise Corpus

This notebook demonstrates a **hybrid** Retrieval-Augmented Generation (RAG) pipeline built on the same OrchestrAI synthetic enterprise corpus as the vector database notebook.

Compared with the basic notebook, this version adds:
- richer document processing through **Docling**
- tokenizer-aware chunking with **HybridChunker**
- an additional safety split before embedding
- a real **Milvus** vector database backend
- explicit source metadata normalization for cleaner retrieval and citations

**Outputs and IDs:** Logged as `implementation="hybrid"` with scenario IDs `h1`–`h5` to `data/results/experimental/hpc_results.csv`. Milvus hybrid retriever **`top_k=10`**. Each run clears only previous `hybrid` rows before saving. Uses collection name `rag_hybrid_docs`. See `README.md` for evaluation.

## What is different from the basic notebook?

### Basic pipeline
- direct DOCX conversion
- simple splitting
- in-memory document store
- simpler indexing flow

### Hybrid pipeline
- richer conversion and chunking
- persistent vector storage in Milvus
- more robust source metadata handling
- a more realistic enterprise-style retrieval pipeline

## Environment requirements

This notebook assumes:
- Python **3.12** (recommended) and `requirements.txt`
- vLLM services are running and reachable from this HPC job/session
- `/nvme/scratch/cy26nk2/llm.env` and `/nvme/scratch/cy26nk2/embd.env` exist with valid API keys, URLs, and model names
- Milvus is available in this environment (the notebook uses a file-backed URI under `notebooks-hpc/`)
- the OrchestrAI corpus under **`data/dummy_data/`** (`PROJECT_ROOT / "data" / "dummy_data"`)
- the first code cell prepends **`src/`** to `sys.path` for `results_logger`

Run the optional install cell once if packages are missing.

## Imports and runtime checks

In [1]:
import sys
import warnings
import time
from pathlib import Path


PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks-hpc" else Path.cwd()
SRC_DIR = PROJECT_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

warnings.filterwarnings("ignore")

print("Python:", sys.executable)

from dotenv import dotenv_values
llm_config = dotenv_values('/nvme/scratch/cy26nk2/llm.env')
embd_config = dotenv_values('/nvme/scratch/cy26nk2/embd.env')
print("LLM model loaded:", llm_config.get("MODEL_NAME", "<missing>"))
print("Embedder model loaded:", embd_config.get("MODEL_NAME", "<missing>"))

from results_logger import save_result_row, clear_results_for_implementation

RESULTS_CSV = PROJECT_ROOT / "data" / "results" / "experimental" / "hpc_results.csv"
# Start this implementation's results from a clean slate.
clear_results_for_implementation("hybrid", results_csv=RESULTS_CSV)

from haystack import Pipeline
from haystack.dataclasses import ChatMessage
from haystack.components.builders import ChatPromptBuilder
from haystack.components.writers import DocumentWriter
from haystack.components.preprocessors import DocumentSplitter

from milvus_haystack import MilvusDocumentStore, MilvusHybridRetriever
from milvus_haystack.function import BM25BuiltInFunction

from haystack.components.embedders import OpenAIDocumentEmbedder, OpenAITextEmbedder
from haystack.components.generators.chat import OpenAIChatGenerator
from haystack.utils import Secret

from haystack_integrations.components.converters.docling import DoclingConverter
from docling.chunking import HybridChunker

print("Imports work")

Python: /nvme/h/cy26nk2/.conda/envs/notebookEnv/bin/python3.14
LLM model loaded: meta-llama/Llama-3.1-8B-Instruct
Embedder model loaded: BAAI/bge-m3
Cleared existing rows for implementation='hybrid'
Imports work


## Connect to Milvus

This notebook uses a file-backed Milvus URI for local persistence in the project workspace.  
This keeps indexing reproducible across reruns in HPC sessions.

In [2]:
COLLECTION_NAME = "rag_hybrid_docs"
db_path = str((PROJECT_ROOT / "notebooks-hpc" / "rag_hybrid.db").resolve())
print("Milvus URI:", db_path)

document_store = MilvusDocumentStore(
    connection_args={"uri": db_path},
    collection_name=COLLECTION_NAME,
    vector_field="vector",
    sparse_vector_field="sparse_vector",
    text_field="text",
    enable_dynamic_field=False,
    builtin_function=BM25BuiltInFunction(
        input_field_names="text",
        output_field_names="sparse_vector",
    ),
    drop_old=True,
)

print("Milvus hybrid collection ready:", COLLECTION_NAME)

Milvus URI: /nvme/h/cy26nk2/Thesis/notebooks-hpc/rag_hybrid.db
Milvus hybrid collection ready: rag_hybrid_docs


## Load the OrchestrAI dataset

The corpus is organized by department under **`data/dummy_data/`**; all `.docx` files are loaded recursively.

In [3]:
DOCUMENTS_DIR = PROJECT_ROOT / "data" / "dummy_data"
FILES = [str(f.resolve()) for f in DOCUMENTS_DIR.rglob("*.docx")]

print("Files found:", len(FILES))
for f in FILES[:10]:
    print("-", Path(f).name)

Files found: 54
- VAL-SALES-002_Webinar_Launch_Brief_refreshed.docx
- VAL-SALES-005_Discount_Exception_Policy_refreshed.docx
- VAL-SALES-001_Lead_Routing_Rules_refreshed.docx
- VAL-SALES-004_CRM_Field_Governance_Note_refreshed.docx
- VAL-SALES-003_Campaign_Approval_Workflow_refreshed.docx
- VAL-LOG-005_Reverse_Logistics_Routing_Memo.docx
- VAL-LOG-004_Regional_Delivery_SLA_Matrix.docx
- VAL-LOG-001_Carrier_Delay_Escalation_Notice.docx
- VAL-LOG-002_Warehouse_Receiving_Checklist.docx
- VAL-LOG-003_Damaged_Goods_Handling_SOP.docx


## Hybrid indexing strategy

To make the advanced pipeline reliable, the indexing flow is:

1. Convert each file with `DoclingConverter`
2. Attach explicit source metadata (`file_name`, `file_path`, `department`)
3. Apply an additional `DocumentSplitter` before embedding
4. Remove metadata fields that Milvus cannot store cleanly
5. Generate embeddings with vLLM-served embedder
6. Write the embedded chunks to Milvus

This file-by-file indexing approach fixes the earlier `unknown source` problem during retrieval.

In [4]:
OLLAMA_EMBED_MODEL = "mxbai-embed-large"
HF_TOKENIZER_ID = "mixedbread-ai/mxbai-embed-large-v1"

converter = DoclingConverter(
    chunker=HybridChunker(
        tokenizer=HF_TOKENIZER_ID,
        max_tokens=300,
    )
)

splitter = DocumentSplitter(
    split_by="word",
    split_length=80,
    split_overlap=20
)

doc_embedder = OpenAIDocumentEmbedder(
    api_key=Secret.from_token(embd_config["VLLM_API_KEY"]),
    api_base_url=f"{embd_config['VLLM_EMBD_URL']}/v1",
    model=embd_config["MODEL_NAME"],
)

writer = DocumentWriter(document_store=document_store)

def normalize_meta(meta):
    cleaned = {}
    for key, value in (meta or {}).items():
        if key == "_split_overlap":
            continue
        if isinstance(value, (str, int, float, bool)) or value is None:
            cleaned[key] = value
        else:
            cleaned[key] = str(value)
    return cleaned

stored_chunks = 0
indexing_start = time.perf_counter()

for i, path in enumerate(FILES, 1):
    print(f"[{i}/{len(FILES)}] Indexing {Path(path).name}")

    converted = converter.run(paths=[path])["documents"]

    for doc in converted:
        meta = normalize_meta(doc.meta)
        meta["file_path"] = path
        meta["file_name"] = Path(path).name
        meta["department"] = Path(path).parent.name
        doc.meta = meta

    split_docs = splitter.run(documents=converted)["documents"]

    for doc in split_docs:
        meta = normalize_meta(doc.meta)
        meta["file_path"] = path
        meta["file_name"] = Path(path).name
        meta["department"] = Path(path).parent.name
        doc.meta = meta

    embedded_docs = doc_embedder.run(documents=split_docs)["documents"]
    writer.run(documents=embedded_docs)

    stored_chunks += len(embedded_docs)

indexing_time_s = round(time.perf_counter() - indexing_start, 4)

print("Indexing completed.")
print("Stored chunks in Milvus:", stored_chunks)
print("Document store count:", document_store.count_documents())
print("Indexing time (s):", indexing_time_s)

[1/54] Indexing VAL-SALES-002_Webinar_Launch_Brief_refreshed.docx


Calculating embeddings: 1it [00:00,  2.19it/s]


[2/54] Indexing VAL-SALES-005_Discount_Exception_Policy_refreshed.docx


Calculating embeddings: 1it [00:00, 28.72it/s]

[3/54] Indexing VAL-SALES-001_Lead_Routing_Rules_refreshed.docx


...

[53/54] Indexing TEAMQA-LOG-001_OrchestrAI_Logistics_and_Field_Fulfillment_Handbook.docx


Calculating embeddings: 100%|██████████| 2/2 [00:01<00:00,  1.56it/s]


[54/54] Indexing TEAMQA-PROD-001_OrchestrAI_Product_and_Program_Management_Handbook.docx


Calculating embeddings: 100%|██████████| 1/1 [00:01<00:00,  1.31s/it]

Indexing completed.
Stored chunks in Milvus: 654
Document store count: 654
Indexing time (s): 34.8671


## Build the hybrid RAG pipeline

The answer-generation logic is aligned with the basic notebook:
- use only the retrieved context
- do not invent missing information
- cite source filenames when possible

In [5]:
template = [
    ChatMessage.from_user(
        """
Respond to the User Query using only the provided Context.

General Guidelines:
- Use only the provided context.
- If the answer is not found in the context, say so clearly.
- Do not make up facts, document sections, pages, or citations.
- If the answer comes from multiple documents, cite all relevant documents.
- Respond in the same language as the user’s query.
- Do not use emojis.
- Be professional and concise.
- Do not write a conclusion or follow-up unless the user asks for it.
- If the context contains a direct policy rule, exception, SLA, or escalation instruction, state it directly and do not hedge.
- Prefer the most specific procedural source over broader handbook summaries when both are present.
- If the retrieved context does not explicitly contain the rule, exception, owner, or next step needed to answer the question, say that the retrieved context is insufficient instead of inferring or guessing.
- Do not mention system names, tools, workflows, or teams unless they are explicitly present in the retrieved context.

Citation Rules:
- For every important claim, cite the source document filename.
- If available in the retrieved context, also mention the document ID or department.
- Do not mention source chapter, section, or page unless they are explicitly present in the retrieved context.

Context:
{% for document in documents %}
[Source: {{ document.meta.get("file_name", document.meta.get("file_path", "unknown source")) }}]
{{ document.content }}

{% endfor %}

Question: {{question}}
Answer:
"""
    )
]

text_embedder = OpenAITextEmbedder(
    api_key=Secret.from_token(embd_config["VLLM_API_KEY"]),
    api_base_url=f"{embd_config['VLLM_EMBD_URL']}/v1",
    model=embd_config["MODEL_NAME"],
)

retriever = MilvusHybridRetriever(
    document_store=document_store,
    top_k=10
)

chat_generator = OpenAIChatGenerator(
    api_key=Secret.from_token(llm_config["VLLM_API_KEY"]),
    api_base_url=f"{llm_config['VLLM_LLM_URL']}/v1",
    model=llm_config["MODEL_NAME"],
    generation_kwargs={
        "temperature": 0.0,
        "top_p": 1.0,
        "seed": 42,
    }
)

prompt_builder = ChatPromptBuilder(
    template=template,
    required_variables=["question", "documents"]
)

advanced_rag_pipeline = Pipeline()
advanced_rag_pipeline.add_component("text_embedder", text_embedder)
advanced_rag_pipeline.add_component("retriever", retriever)
advanced_rag_pipeline.add_component("prompt_builder", prompt_builder)
advanced_rag_pipeline.add_component("llm", chat_generator)

advanced_rag_pipeline.connect("text_embedder.embedding", "retriever.query_embedding")
advanced_rag_pipeline.connect("retriever.documents", "prompt_builder.documents")
advanced_rag_pipeline.connect("prompt_builder.prompt", "llm.messages")

🚅 Components
  - text_embedder: OpenAITextEmbedder
  - retriever: MilvusHybridRetriever
  - prompt_builder: ChatPromptBuilder
  - llm: OpenAIChatGenerator
🛤️ Connections
  - text_embedder.embedding -> retriever.query_embedding (list[float])
  - retriever.documents -> prompt_builder.documents (List[Document])
  - prompt_builder.prompt -> llm.messages (list[ChatMessage])

## Helper function for hybrid queries

The function below:
- embeds the question
- retrieves the top document chunks
- prints the retrieved hybrid sources and snippets
- runs the full hybrid RAG pipeline
- prints the final answer

In [6]:
def run_hybrid_query(question, top_k=10):
    print("=" * 80)
    print("QUESTION:")
    print(question)
    print("=" * 80)

    retrieval_start = time.perf_counter()

    q_emb = text_embedder.run(text=question)["embedding"]

    retr_out = retriever.run(
        query_embedding=q_emb,
        query_text=question,
        top_k=top_k,
    )
    top_docs = retr_out["documents"]

    retrieval_time_s = round(time.perf_counter() - retrieval_start, 4)

    print(f"\nRETRIEVED DOCUMENTS: {len(top_docs)}")
    print(f"RETRIEVAL TIME (s): {retrieval_time_s}\n")

    for i, d in enumerate(top_docs, 1):
        source = d.meta.get("file_name", d.meta.get("file_path", "unknown source"))
        department = d.meta.get("department", "unknown department")
        snippet = (d.content or "")[:400].replace("\n", " ")
        print(f"[{i}] Source: {source} | Department: {department}")
        print(f"    {snippet}...")
        print()

    generation_start = time.perf_counter()

    result = advanced_rag_pipeline.run({
        "text_embedder": {"text": question},
        "retriever": {"query_text": question, "top_k": top_k},
        "prompt_builder": {"question": question},
    })

    answer = result["llm"]["replies"][0].text
    generation_time_s = round(time.perf_counter() - generation_start, 4)

    print("=" * 80)
    print("FINAL ANSWER:")
    print("=" * 80)
    print(answer)
    print(f"\nGENERATION TIME (s): {generation_time_s}")

    return top_docs, answer, retrieval_time_s, generation_time_s

In [7]:
def run_and_save_hybrid(question, scenario_id, top_k=10):
    top_docs, answer, retrieval_time_s, generation_time_s = run_hybrid_query(
        question,
        top_k=top_k,
    )

    save_result_row(
        implementation="hybrid",
        scenario_id=scenario_id,
        question=question,
        generated_answer=answer,
        top_docs=top_docs,
        retriever_top_k=top_k,
        reranker_top_k=None,
        indexing_time_s=indexing_time_s,
        retrieval_time_s=retrieval_time_s,
        generation_time_s=generation_time_s,
        results_csv=RESULTS_CSV,
    )

    return top_docs, answer

## Evaluation scenarios

The following five questions are used as progressively more complex RAG evaluation scenarios:

1. Basic fact retrieval
2. Procedural retrieval
3. Multi-team operational reasoning
4. Policy + exception handling
5. Cross-functional enterprise reasoning

### Scenario 1 — Basic fact retrieval
This is the clean baseline. It shows whether the system can retrieve one exact operational fact correctly.
- Single-document, single-fact retrieval.
- Tests basic embedding-based retrieval accuracy.
- Expected answer should come from the Customer Success / Support materials.

In [8]:
top_docs_hybrid_s1, answer_hybrid_s1 = run_and_save_hybrid(
    'What is the first-response SLA for a Sev-1 support incident?',
    scenario_id="h1",
    top_k=10,
)

QUESTION:
What is the first-response SLA for a Sev-1 support incident?

RETRIEVED DOCUMENTS: 10
RETRIEVAL TIME (s): 0.605

[1] Source: TEAMQA-CS-001_OrchestrAI_Customer_Success_and_Support_Handbook.docx | Department: Q&A Hanbooks
    Customer Success and Support Handbook Service targets and escalation guide Sev-1 first response, **Target** = 15 min. Sev-1 first response, **Reviewed by** = Support manager. Sev-1 first response, **Notes** = Clock starts when the case is correctly classified. Sev-2 first response, **Target** = 1 hour. Sev-2 first response, **Reviewed by** = Support manager. Sev-2 first response, **Notes** = Inclu...

[2] Source: TEAMQA-CS-001_OrchestrAI_Customer_Success_and_Support_Handbook.docx | Department: Q&A Hanbooks
    Customer Success and Support Handbook Specialised Q&A **Q: Who owns customer communication during a Sev-1 incident?** **A:** The assigned support lead owns the operational updates, but the Customer Success Manager owns stakeholder alignment and execu

### Scenario 2 — Procedural retrieval
This scenario is more complex because the answer is not a single fact, but a short operational process.
- Procedure and checklist retrieval.
- Tests whether the system can retrieve and summarize structured steps clearly.
- Expected answer should come mainly from the HR / People Operations materials.

In [9]:
top_docs_hybrid_s2, answer_hybrid_s2 = run_and_save_hybrid(
    'What steps must be completed before a new employee is considered ready for day one at OrchestrAI?',
    scenario_id="h2",
    top_k=10,
)

QUESTION:
What steps must be completed before a new employee is considered ready for day one at OrchestrAI?

RETRIEVED DOCUMENTS: 10
RETRIEVAL TIME (s): 0.5895

[1] Source: VAL-HR-001_Day_One_Onboarding_Checklist.docx | Department: HR
    Day-One Onboarding Checklist **Document ID:** VAL-HR-001 **Owner:** HR & People Operations **Confidentiality:** Internal Use Only Purpose: This checklist standardizes the minimum tasks that must be complete before an OrchestrAI employee can be considered ready for day-one onboarding. **Department**, **Value** = HR & People Operations. **Effective date**, **Value** = 2026-02-24. **Applies to**, **V...

[2] Source: TEAMQA-HR-001_OrchestrAI_HR_and_People_Operations_Handbook.docx | Department: Q&A Hanbooks
    HR and People Operations Handbook Specialised Q&A **Q: What does HR and People Operations own at OrchestrAI?** **A:** The team owns employee record quality, onboarding coordination, policy communication, leave administration, manager guidance, workp

### Scenario 3 — Multi-team operational reasoning
This scenario requires combining information from more than one team to answer a realistic workplace problem.
- Multi-document, cross-team retrieval.
- Tests whether the system can synthesize HR, IT, and Security responsibilities.
- Expected answer should identify both the involved teams and the next actions.

In [10]:
top_docs_hybrid_s3, answer_hybrid_s3 = run_and_save_hybrid(
    'A new engineer starts on Monday, but their laptop has not arrived and their required access is still pending. Which teams are involved and what should happen next?',
    scenario_id="h3",
    top_k=10,
)

QUESTION:
A new engineer starts on Monday, but their laptop has not arrived and their required access is still pending. Which teams are involved and what should happen next?

RETRIEVED DOCUMENTS: 10
RETRIEVAL TIME (s): 0.5871

[1] Source: VAL-IT-001_Laptop_Provisioning_FAQ.docx | Department: IT
    Common Questions **Q: How far in advance should a laptop be requested?** A: Standard new-hire laptop requests should be present in the hiring workflow at least seven business days before the employee start date. International shipments, executive hires, and custom software bundles may require ten to fifteen business days. **Q: Which team starts the request?** A: The request begins with the hiring ...

[2] Source: TEAMQA-PROD-001_OrchestrAI_Product_and_Program_Management_Handbook.docx | Department: Q&A Hanbooks
    involved?** A. PMO coordinates the master communications plan, but each functional owner remains responsible for the accuracy of their audience-specific content. **Q. What evidence

### Scenario 4 — Policy + exception handling
This scenario checks whether the system can distinguish the normal rule from the emergency exception path.
- Policy retrieval with exception reasoning.
- Tests whether the system can identify when a standard workflow may be bypassed.
- Expected answer should combine Finance / Procurement rules with operational urgency.

In [11]:
top_docs_hybrid_s4, answer_hybrid_s4 = run_and_save_hybrid(
    'Can OrchestrAI make an urgent hardware purchase without following the normal PO workflow if a warehouse outage is at risk?',
    scenario_id="h4",
    top_k=10,
)

QUESTION:
Can OrchestrAI make an urgent hardware purchase without following the normal PO workflow if a warehouse outage is at risk?

RETRIEVED DOCUMENTS: 10
RETRIEVAL TIME (s): 0.5831

[1] Source: TEAMQA-FIN-001_OrchestrAI_Finance_and_Procurement_Handbook.docx | Department: Q&A Hanbooks
    Finance and Procurement Handbook Specialised Q&A **Q: What does Finance and Procurement own at OrchestrAI?** **A:** The team owns vendor onboarding, purchase-order governance, invoice and payment controls, employee reimbursement policy, budget coordination, and month-end readiness. It is also the main gatekeeper for spend discipline and exception documentation. **Q: When is a purchase order requir...

[2] Source: TEAMQA-IT-001_OrchestrAI_IT_Workplace_and_Service_Desk_Handbook.docx | Department: Q&A Hanbooks
    reviewed?** **A:** Baseline standards are reviewed quarterly or after a major security, OS, or vendor change. Emergency policy updates may be issued outside the normal cadence when risk warr

### Scenario 5 — Cross-functional enterprise reasoning
This is the most complex scenario because it requires cross-functional reasoning across several business units.
- Multi-hop, multi-document retrieval.
- Tests whether the system can combine support, product, logistics, and commercial risk signals.
- Expected answer should identify the teams involved and propose a coordinated next-step plan.

In [12]:
top_docs_hybrid_s5, answer_hybrid_s5 = run_and_save_hybrid(
    'A customer renewal is at risk because support cases are recurring, a requested feature is still unavailable, and replacement hardware is delayed. What should OrchestrAI do next, and which teams must be involved?',
    scenario_id="h5",
    top_k=10,
)

QUESTION:
A customer renewal is at risk because support cases are recurring, a requested feature is still unavailable, and replacement hardware is delayed. What should OrchestrAI do next, and which teams must be involved?

RETRIEVED DOCUMENTS: 10
RETRIEVAL TIME (s): 0.5839

[1] Source: TEAMQA-LOG-001_OrchestrAI_Logistics_and_Field_Fulfillment_Handbook.docx | Department: Q&A Hanbooks
    OrchestrAI Logistics and Field Fulfillment Handbook 5. Operational Questions and Answers What data must be preserved for auditability? At minimum, the team must preserve who requested the movement, who approved it, which serial numbers were involved, when the shipment left the warehouse, which carrier handled it, and what exception decisions were made. Auditability is a first-class requirement bec...

[2] Source: TEAMQA-IT-001_OrchestrAI_IT_Workplace_and_Service_Desk_Handbook.docx | Department: Q&A Hanbooks
    Team Q&A **Q: What does the IT team own at OrchestrAI?** **A:** IT owns end-user computing, c